# Field mapping

Projecting points $p$ from tangent space to a (quadratic bezier) curve $S$ and building SDF-based field.

- SDF (2D):
    - signed distance from sample to curve
    - signed (parametric) arc distance along curve from apex to projection
- Flow (3D):
    - tangent flow at projection
- Height (1D):
    - signed elevation of corresponding curve point over plane

#### Closest point

- $p$ — sample point in UV space
- $p^* = S(t^*)$ — nearest point on curve

$$
t^* = \argmin_t{||p - S(t)||^2}
$$

#### General solution

- $S(t)$ — general parametric curve
- $F(t) = S'(t)$ ­— first derivative, velocity/flow
- $v(t) = S(t) - p$ — vector point to curve

$$
(p - S(t))·F(t) = 0
\tag{perpendicularity}
$$

$$
\frac{dv^2}{dt}(t) = 0
\tag{min disquance}
$$

The equations are equivalent and almost identical.

Solving for $t$ gives points with locally minimum distance.
Globally minimum should be chosen by additional computations.

#### Distance

- $v_p = p - p^*$ — vector from a projection point to the sample

$$D^2 = ||v_p||^2 = v_p·v_p$$

Doesn't make sense to squaeroot because it can be used as is.

#### Orientation

- $v = p - p^*$ — vector from a projection point to the sample
- $f = S'(t^*)$ ­— tangential flow at the projection point
- $z = (v × f)_z = v_x f_y - v_y f_x$ — z-component of cross product of 2D vectors into 3D

$$
\begin{cases}
z > 0 \implies \text{at right-hand side} \\
z < 0 \implies \text{at left-hand side} \\
z = 0 \implies \text{exact on the curve, should not happen}\\
\end{cases}
$$

Numeric value $z = |v||f|\sin\phi$


In [ ]:
from typing import NamedTuple
from collections.abc import Callable

import math
import numpy as np
import numpy.random as nprand
from numpy.typing import NDArray

Vec = np.ndarray[tuple[int], np.dtype[np.floating]]  # 2D or 3D
vec = lambda *args: np.array(args, dtype=float)
asarr = np.asarray
nparr = lambda *args: np.array(args)
npgen = lambda iter: np.fromiter(iter, dtype=float)
dot = lambda v1, v2: float(v1 @ v2)
dots = lambda a1, a2: (a1 * a2).sum(axis=-1)
powers = lambda t: np.array([np.ones(np.shape(t)), t, np.square(t)]).T

In [ ]:
from field import Field, save_field

In [ ]:
def gridspace(lowbound: float, highbound: float, res: int) -> NDArray:
    """Generate XY sampling space with coordinates in centers of grid cells (pixels)
    Returns array of 2d coords (res×res×2)
    """
    r = (highbound - lowbound) / res / 2  # pixel half-size
    cspace = np.linspace(lowbound, highbound, res, endpoint=False) + r
    X, Y = np.meshgrid(cspace, cspace)
    return np.stack((X, Y)).T

In [ ]:
FIX00 = vec(0.0, 0.0)
FIX11 = vec(1.0, 1.0)


def random_points2d(num: int, beg=FIX00, end=FIX11):
    """Geneates control points from fixed corners (1, 0) (0, 1) and distorted midpoints"""
    xx = np.linspace(beg[0], end[0], num)
    yy = np.linspace(beg[1], end[1], num)

    j = 1.0 / num
    coords = np.stack((xx, yy)).T
    coords[1:-1] += np.random.randn(num - 2, 2) * j
    return coords


def random_points3d(num: int, beg=FIX00, end=FIX11):
    """Geneates control points from fixed corners (1, 0, 0) (0, 1, 0) and distorted midpoints"""
    xx = np.linspace(beg[0], end[0], num)
    yy = np.linspace(beg[1], end[1], num)
    zz = np.zeros(num)

    j = 1.0 / num
    coords = np.stack((xx, yy, zz)).T
    coords[1:-1] += np.random.randn(num - 2, 3) * j
    return coords

In [ ]:
def disquance(p0: Vec, p1: Vec):
    v = p1 - p0
    return dot(v, v)


def disquances(p0: Vec, pp: NDArray):
    vv = pp - p0
    return (vv * vv).sum(axis=1)


def orientation(frw: Vec, vec: Vec):
    return np.sign(vec[0] * frw[1] - vec[1] * frw[0])


def orientations(frws: NDArray, vecs: NDArray):
    return np.sign(vecs[:, 0] * frws[:, 1] - vecs[:, 1] * frws[:, 0])

## Quadratic Beziel

A segment of a B2-Spline over points is equivalent to B2zier througn midpoints.

- $c_1 = \frac{p_1 + p_2}{2}$
- $c_2 = p_2$
- $c_3 = \frac{p_3 + p_2}{2}$


In [ ]:
def split_controls(controls: NDArray) -> tuple[NDArray, NDArray, NDArray]:
    """Split long sequence of spline control points into segments
    Returns: 3 arrays of p1 p2 p3 for each segment"""
    num = controls.shape[0]
    return (
        controls[0 : num - 2],
        controls[1 : num - 1],
        controls[2 : num - 0],
    )


def beziefy_controls(p1: NDArray, p2: NDArray, p3: NDArray) -> tuple[NDArray, NDArray, NDArray]:
    """Convert b2spline control points to b2zier controls
    Returns: c1 c2 c3 (arrays or singular points)"""
    return (p1 + p2) * 0.5, p2, (p2 + p3) * 0.5

### Parametrics

Localized around its apex point: $S(0) = c_a$

$$
S_{world}(t) = c_a + S(t)
$$

- $v_1 = c_1 - c_2$
- $v_3 = c_3 - c_2$
- $v_a = v_3 + v_1$ — parabolic axial vector
- $v_t = v_3 - v_1$ — parabolic tangential vector
- $c_a = c_2 + .25 v_a$ — apex point

$$
S(t) - c_a=
\big[1, t, t^2 \big]
\begin{bmatrix}
0 \\
v_t \\
v_a \\
\end{bmatrix}
$$

$$
\frac{dS}{dt}(t) =
\big[1, t, t^2 \big]
\begin{bmatrix}
v_t \\
2 v_a \\
0 \\
\end{bmatrix}
$$


Possible special cases

- $v_1 = 0, v_3 = 0$ all collapsed to single point $c_2$
- $v_1 = 0, v_3 \ne 0 \implies v_a = v_3, v_t = v_3$ still valid
- $v_1 \ne 0, v_3 = 0 \implies v_a = v_1, v_t = -v_1$ still valid
- $v_1 + v_3 = 0 \implies v_a = 0, v_t = -2 v_3, c_a = c_2$ linear case $S(t) = c_2 - 2 v_3 t$
- $v_1 = v_3 \implies v_t = 0, v_a = 2 v_3$ collapsed parabola, $S(t) = c_a + 2 v_3 t^2$ still valid


In [ ]:
class Params(NamedTuple):
    ca: Vec
    vt: Vec
    va: Vec

    @property
    def dim(self):
        return self.ca.shape[-1]


def beziel_params(c1: Vec, c2: Vec, c3: Vec) -> Params:
    """Calculate parabolic basis ca vt va"""
    va = c3 + c1 - 2 * c2
    vt = c3 - c1
    ca = c2 + 0.25 * va
    return Params(ca, vt, va)


def beziel_M(params: Params):
    return np.array([np.zeros(params.dim), params.vt, params.va])


def beziel_Mdt(params: Params):
    return np.array([params.vt, 2 * params.va, np.zeros(params.dim)])

In [ ]:
# all localized aound c_a


def beziel_curve(params: Params, t: float) -> Vec:
    return powers(t) @ beziel_M(params)


def beziel_flow(params: Params, t: float) -> Vec:
    return powers(t) @ beziel_Mdt(params)


Vectorized = Callable[[Params, NDArray], NDArray]
nd_beziel_curve: Vectorized = np.vectorize(beziel_curve, signature="() -> (d)", excluded={0})
nd_beziel_flow: Vectorized = np.vectorize(beziel_flow, signature="() -> (d)", excluded={0})

---


In [ ]:
# from widgets import CurveDualView

In [ ]:
# view = CurveDualView()
# display(view)

In [ ]:
# points = random_points3d(3)
# controls = beziefy_controls(*points)
# params = beziel_params(*controls)

In [ ]:
# tspace = np.linspace(-0.5, 0.5, 17)
# view.controls = points
# view.tspace = tspace
# view.curve = nd_beziel_curve(params, tspace) + params.ca
# view.flow = nd_beziel_flow(params, tspace)

---


# Projecting

- $p_a = p - c_a$ — point relative to apex
- $v_p(t)|p = p - S(t)$ ­— vector from curve to sample point
- $v_p^2 = v_p·v_p = |v_p|^2$ — squared distance from curve to point

Equation for local minima/maxima:

$$
\frac{dv_p^2}{dt} =
[1, t, t^2, t^3]
\begin{pmatrix}
-p_a v_t \\
-2 p_a v_a + v_t^2 \\
3 v_a v_t \\
2 v_a^2 \\
\end{pmatrix}
= 0
$$

There are 1 up to 3 bending points. Need to select the best.


Edge cases:

- $v_a = 0, v_t = 0$ all collapsed to one point
    - $t = 0$
- $v_a = 0, v_t \ne 0$ collapsed to line, still valid
    - $v_t^2 t - p_a v_t = 0$
    - $t = \frac{p_a v_t}{v_t^2}$
- $v_a \ne 0, v_t = 0$ squized to line
    - $-2p_a v_a t + 2 v_a^2 t^2 = 0$
    - $t_0 = 0$
    - $-2p_a v_a + 2 v_a^2 t = 0$
    - $t = \frac{p_a v_a}{v_a^2}$


In [ ]:
from fqs import single_cubic, single_quadratic  # fast

realroots = lambda roots: {r.real for r in roots if r.imag == 0}

$$a_3 t^3 + a_2 t^2 + a_1 t + a_0 = 0$$

Special cases

- $a_3 t^3 + a_0 = 0$ // $a_1 = a_2 = 0$
    - $t = \sqrt[3]{-a_0/a_3}$
- $a_3 t^3 = 0$ // $a_2 = a_1 = a_0 = 0$
    - $t=0$
- $a_3 t^3 + a_1 t = 0$ // $a_2 = a_0 = 0$
    - $t_0 = 0$
    - $a_3 t^2 + a_1 = 0$
    - $a_1/a_3 \lt 0$
        - $t = \sqrt{-a_1/a_3}$
- $a_3 t^3 + a_2 t^2 = 0$ // $a_1 = a_0 = 0$
    - $t_0 = 0$
    - $a_3 t + a_2 = 0$
    - $t = -a_2/a_3$
- $a_3 t^3 + a_2 t^2 + a_1 t = 0$ // $a_0 = 0$
    - $t_0 = 0$
    - $a_3 t^2 + a_2 t + a_1 = 0$
    - quadratic subcase


In [ ]:
def solve_equation(vt: Vec, va: Vec, pa: Vec) -> set[float]:
    """Solve the equation of perpendicularity = extrema of square distance
    Return all real roots
    """
    dot_aa = dot(va, va)
    dot_at = dot(va, vt)
    dot_tt = dot(vt, vt)
    dot_pa = -dot(pa, va)
    dot_pt = -dot(pa, vt)

    za, zt = np.isclose([dot_aa, dot_tt], 0.0)

    if za and zt:
        return {0.0}
    elif za:
        return {dot_pt / dot_tt}
    elif zt:
        return {0.0, dot_pa / dot_aa}

    a0 = dot_pt
    a1 = dot_tt + 2 * dot_pa
    a2 = 3 * dot_at
    a3 = 2 * dot_aa

    z0, z1, z2 = np.isclose([a0, a1, a2], 0.0)

    if z0:
        if z1 and z2:
            return {0.0}
        elif z1:
            return {0.0, -a2 / a3}
        elif z2:
            roots = {0.0}
            a13 = a1 / a3
            if a13 < 0:
                roots.add(math.sqrt(-a13))
            return roots
        else:
            roots = {0.0}
            roots |= realroots(single_quadratic(a3, a2, a1))
            return roots
    elif z1 and z2:
        return {math.cbrt(-a0 / a3)}
    else:
        return realroots(single_cubic(a3, a2, a1, a0))

In [ ]:
def projection(params: Params, sample: Vec) -> tuple[float, Vec, float, Vec, float]:
    """Project a single point to curve and get all the characteristics"""
    # assert params.dim == 2

    pa = sample - params.ca
    tt = solve_equation(params.vt, params.va, pa)
    tt = npgen(tt)

    pnts = nd_beziel_curve(params, tt)
    vecs = pa - pnts
    dsqs = dots(vecs, vecs)

    best = np.argmin(dsqs)
    arc = tt[best]
    pnt = pnts[best]
    vec = vecs[best]
    dsq = dsqs[best]
    flw = beziel_flow(params, arc)
    sgn = orientation(flw, vec)

    return arc, params.ca + pnt, dsq, flw, sgn


# vectorized over array of sample points
np_projection = np.vectorize(projection, signature="(d) -> (),(d),(),(d),()", excluded={0})

In [ ]:
def projections_all(params: Params, sample: Vec) -> tuple[NDArray, NDArray, NDArray, NDArray, NDArray]:
    """Project a single point to curve and get all the characteristics"""
    # assert params.dim == 2

    pa = sample - params.ca
    tt = solve_equation(params.vt, params.va, pa)
    tt = npgen(tt)

    pnts = nd_beziel_curve(params, tt)
    vecs = pa - pnts
    dsqs = dots(vecs, vecs)
    flws = nd_beziel_flow(params, tt)
    sgns = orientations(flws, vecs)

    return tt, params.ca + pnts, dsqs, flws, sgns

In [ ]:
from debugfield import debug_field

from widgets import Field2DView, Field3DView, FieldDualView

## Single Bezier


In [ ]:
def fieldmap2d(params: Params, samples: NDArray) -> Field:
    assert params.dim == 2
    arc, pnt, dsq, flw, sgn = np_projection(params, samples)
    return Field(samples, arc, pnt, dsq, flw, sgn)


def fieldmap3d(params: Params, samples: NDArray) -> Field:
    assert params.dim == 3
    samples = np.dstack((samples, np.zeros(samples.shape[:-1])))
    arc, pnt, dsq, flw, sgn = np_projection(params, samples)
    return Field(samples, arc, pnt, dsq, flw, sgn)


def fieldmap2d3(params2d: Params, params3d: Params, samples: NDArray) -> Field:
    assert params2d.dim == 2
    assert params3d.dim == 3
    arc2, _pnt2, dsq2, _flw2, sgn2 = np_projection(params2d, samples)
    pnt3 = nd_beziel_curve(params3d, arc2)  # could only process z
    flw3 = nd_beziel_flow(params3d, arc2)  # could only process z
    return Field(samples, arc2, pnt3, dsq2, flw3, sgn2)


def pickmap2d(params: Params, sample: Vec) -> Field:
    assert params.dim == 2
    arc, pnt, dsq, flw, sgn = projections_all(params, sample)
    samples = np.full((*arc.shape, sample.shape[0]), sample)
    return Field(samples, arc, pnt, dsq, flw, sgn)

---


In [ ]:
view = FieldDualView()
display(view)

In [ ]:
tspace = np.linspace(-0.5, 0.5, 33)
gspace = gridspace(0.0, 1.0, 32)
view.tspace = tspace
view.gspace = gspace

c1, c2, c3 = random_points3d(3)
params3d = beziel_params(c1, c2, c3)
params2d = beziel_params(c1[:2], c2[:2], c3[:2])

In [ ]:
view.controls = nparr(c1, c2, c3)
view.curve = nd_beziel_curve(params3d, tspace) + params3d[0]
view.field = fieldmap2d3(params2d, params3d, gspace)

ca, vt, va = params3d
view.axes = (["Va", "Ca", "Vt"], nparr(ca + va, ca, ca + vt))

In [ ]:
def on_pick(sample: Vec):
    view.poke = pickmap2d(params2d, sample)


view.observe(lambda change: on_pick(change.new), "pick")

## TODO: Continous spline

Filter out segments by horizon visibility


### Visibility

$$
\begin{cases}
(p - c_2) · v_1 > v_1 · v_1 \implies t^* < -0.5
\\
(p - c_2) · v_3 > v_3 · v_3 \implies t^* > +0.5
\end{cases}
$$

Indicates that a sample projection guarantied to lie behind horizons of segment
